# This notebook applies the Statistical Methods on the data for fraud and hoarding detection, revenue audit to discover the key business insights 

## Pillar 1: Probability Distributions & Descriptive Stats.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

def execute_pillar_1():
    print(">>> INITIALIZING PILLAR 1: DESCRIPTIVE STATS & PROBABILITY...")


    data_folder = os.path.join('..', 'Data')
    processed_folder = os.path.join(data_folder, 'Processed')
    
    # Point to the cleanly named database
    db_path = os.path.join(data_folder, 'PCCMD_Warehouse_New2.db')

    # Check if the database exists before trying to connect
    if not os.path.exists(db_path):
        print(f"!!! CRITICAL ERROR: Database not found at {db_path}")
        return  # FIX: You must halt execution if the DB is missing
    
    conn = sqlite3.connect(db_path)
    print(">>> CONNECTION SUCCESSFUL: Linked to PCCMD Warehouse.")
    
    # 2. Extract the Enriched Telemetry
    # We want the Magistrate ID, Time, and the SQL-calculated Minutes Delta
    query = """
        SELECT 
            Magistrate_ID, 
            Date, 
            Time, 
            Minutes_Since_Last_Inspection 
        FROM Fact_Inspections_Enriched;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    # 3. IDENTIFY THE NIGHT OWLS (Time-based Anomaly Detection)
    # The SLA explicitly bans inspections between 22:00 (10 PM) and 05:00 (5 AM)
    df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour
    df['Is_Night_Owl_Violation'] = np.where((df['Hour'] >= 22) | (df['Hour'] < 5), 1, 0)

    # 4. IDENTIFY THE SPEEDRUNNERS (Probability & SLA Cooldown Violation)
    # The SLA mandates an 8-minute cooldown.
    df['Is_Speedrunner_Violation'] = np.where(df['Minutes_Since_Last_Inspection'] < 8, 1, 0)

    # 5. DESCRIPTIVE STATISTICS: Profiling the Magistrates
    print(">>> Calculating Statistical Profiles per Magistrate...")
    
    # We group by Magistrate to calculate their unique statistical footprint
    magistrate_profile = df.groupby('Magistrate_ID').agg(
        Total_Inspections=('Magistrate_ID', 'count'),
        Night_Owl_Violations=('Is_Night_Owl_Violation', 'sum'),
        Speedrunner_Violations=('Is_Speedrunner_Violation', 'sum'),
        Mean_Minutes_Between_Inspections=('Minutes_Since_Last_Inspection', 'mean'),
        Std_Dev_Minutes=('Minutes_Since_Last_Inspection', 'std')
    ).reset_index()

    # 6. CALCULATING THE FRAUD RATE
    magistrate_profile['Fraud_Rate_%'] = round(
        ((magistrate_profile['Night_Owl_Violations'] + magistrate_profile['Speedrunner_Violations']) 
         / magistrate_profile['Total_Inspections']) * 100, 2
    )

    # Sort by the most anomalous magistrates (Highest Fraud Rate)
    bad_actors = magistrate_profile.sort_values(by='Fraud_Rate_%', ascending=False)

    # 7. EXPORT THE PROOF
    # Dynamically route the output to the Processed folder with the correct naming convention
    os.makedirs(processed_folder, exist_ok=True)
    output_path = os.path.join(processed_folder, "Statistical_Pillar1_Magistrate_Fraud_Profiles.csv")
    bad_actors.to_csv(output_path, index=False)
    
    # 8. THE EXECUTIVE SUMMARY (Print to terminal)
    print("\n" + "="*70)
    print("EXECUTIVE SUMMARY: PILLAR 1 (SLA COMPLIANCE)")
    print("="*70)
    print(f"Total Magistrates Audited: {len(magistrate_profile)}")
    
    # Let's show the top 5 highest-risk profiles to prove the math works
    print("\n>>> TOP 5 HIGHEST-RISK MAGISTRATE PROFILES:")
    print(bad_actors.head(5).to_string(index=False))
    
    print("\n" + "="*70)
    print(f"SUCCESS: Pillar 1 Complete. Fraud Profiles exported to:\n{output_path}")
    print("="*70)

if __name__ == "__main__":
    execute_pillar_1()

>>> INITIALIZING PILLAR 1: DESCRIPTIVE STATS & PROBABILITY...
>>> CONNECTION SUCCESSFUL: Linked to PCCMD Warehouse.
>>> Calculating Statistical Profiles per Magistrate...

EXECUTIVE SUMMARY: PILLAR 1 (SLA COMPLIANCE)
Total Magistrates Audited: 145

>>> TOP 5 MOST ANOMALOUS MAGISTRATES (THE BAD ACTORS):
Magistrate_ID  Total_Inspections  Night_Owl_Violations  Speedrunner_Violations  Mean_Minutes_Between_Inspections  Std_Dev_Minutes  Fraud_Rate_%
      MAG-205                303                   275                       0                         21.000000         5.354126         90.76
      MAG-105                333                   290                       0                         30.000000              NaN         87.09
      MAG-410               1085                     0                     461                          2.021692         0.831164         42.49
      MAG-310               1085                     0                     449                          2.000000        

## Pillar 2: Hypothesis Testing & P-Values - Check if bad governance leads to expensive food


### This code answers the following questions:

1. Does Enforcement "Speedrunning" Predict Price Spikes?
The script calculates a Daily Fraud Rate based on two specific "red flag" behaviors:

•	Speed Violations: Were inspections logged less than 8 minutes apart? (Suggesting the inspector didn't actually visit the site).

•	Night Violations: Were inspections logged during "ghost hours" (10 PM – 5 AM)?

•	The Question: Is there a statistically significant correlation between these high-fraud days and the price of onions in that city a few days later?

###
###
###

2. What is the "Predictive Sweet Spot" (Lag Time)?

Because market prices don't react instantly to a lack of enforcement, the code uses Multi-Lag Correlation Analysis.

•	The Question: How many days after a "systemic collapse" in inspection quality does the market price peak? Is the signal strongest at 1, 3, 5, or 7 days?

###
###
###
###

3. Is the Relationship Statistically Significant or Just Noise?
Using a Welch’s T-Test (which handles groups with different sizes and variances), the code compares two distinct environments:

•	Gold Standard: Days/cities with 0% fraud.

•	Systemic Collapse: Days/cities with >15% fraud.

•	The Question: Is the price difference between these two groups "real" ($p < 0.05$), or could it have happened by random chance?

###

4. What is the "Inflationary Delta" of Corruption?

The script calculates the average price of onions 3 days after honest inspections vs. 3 days after fraudulent ones.

•	The Question: In PKR (or the local currency), exactly how much more expensive do onions become when magistrates stop doing their jobs properly? (e.g., "Observed Inflationary Delta: +12.45%").

###

5. What should the Operational Response be?
The "Brutal Verdict" section provides a binary decision for leadership:

•	If $p < 0.05$: The problem is market manipulation (hoarding) made possible by bad enforcement. Action: Deploy anti-hoarding squads.

•	If $p > 0.05$: The problem is likely supply chain or logistics, and the bad inspections are just an HR issue. Action: Focus on supply chain management.



In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
import os
import warnings

warnings.filterwarnings('ignore')

def execute_pillar_2_hardened():
    print(">>> INITIALIZING PILLAR 2: LAGGED CORRELATION & ROBUST HYPOTHESIS TESTING...")

    # --- RELATIVE PATH DEFINITIONS ---
    # Navigate up from /Notebooks to the /Data folder
    data_folder = os.path.join('..', 'Data')
    processed_folder = os.path.join(data_folder, 'Processed')
    
    # Point to the cleanly named database
    db_path = os.path.join(data_folder, 'PCCMD_Warehouse_New2.db')

    # 1. Secure Connection Logic
    if not os.path.exists(db_path):
        print(f"!!! CRITICAL ERROR: Database not found at {db_path}")
        return
    
    conn = sqlite3.connect(db_path)

    # 2. ROBUST DATA EXTRACTION
    # Fixed SQL Logic: Using strftime to handle '9:30:00' vs '09:30:00' variations
    fraud_query = """
        SELECT  
            Date, 
            City,
            COUNT(Inspection_ID) as Total_Inspections,
            SUM(CASE WHEN Minutes_Since_Last_Inspection < 8 THEN 1 ELSE 0 END) as Speed_Violations,
            SUM(CASE WHEN strftime('%H', Time) >= '22' OR strftime('%H', Time) < '05' THEN 1 ELSE 0 END) as Night_Violations
        FROM Fact_Inspections_Enriched
        GROUP BY Date, City;
    """
    
    market_query = "SELECT Date, City, Onion_Price FROM Fact_Market;"
    
    fraud_df = pd.read_sql_query(fraud_query, conn)
    market_df = pd.read_sql_query(market_query, conn)
    conn.close()

    # 3. DATE PREPARATION & LAGGED TARGETS (Addressing the Endogeneity Trap)
    # We create a 'Price_3_Days_Later' column to see if fraud TODAY predicts inflation LATER
    market_df['Date'] = pd.to_datetime(market_df['Date'])
    fraud_df['Date'] = pd.to_datetime(fraud_df['Date'])
    
    market_df = market_df.sort_values(['City', 'Date'])
    market_df['Price_Lead_3D'] = market_df.groupby('City')['Onion_Price'].shift(-3)

    # 4. MASTER MERGE
    analysis_df = pd.merge(fraud_df, market_df, on=['Date', 'City'], how='inner')
    analysis_df['Daily_Fraud_Rate'] = (analysis_df['Speed_Violations'] + analysis_df['Night_Violations']) / analysis_df['Total_Inspections']

    # 5. TERNARY CLASSIFICATION (Replacing Arbitrary 5% Threshold)
    def classify_integrity(rate):
        if rate == 0: return 'Gold Standard (0%)'
        elif rate <= 0.15: return 'Operational Noise (1-15%)'
        else: return 'Systemic Collapse (>15%)'

    analysis_df['Integrity_Tier'] = analysis_df['Daily_Fraud_Rate'].apply(classify_integrity)

    # --- NEW SECTION: MULTI-LAG CORRELATION ANALYSIS ---
    print("\n>>> CALCULATING CROSS-LAG CORRELATION MATRIX...")
    
    # 1. Create multiple lags to find the 'Predictive Sweet Spot'
    lag_days = [1, 3, 5, 7]
    for lag in lag_days:
        col_name = f'Price_Lead_{lag}D'
        analysis_df[col_name] = analysis_df.sort_values(['City', 'Date']).groupby('City')['Onion_Price'].shift(-lag)

    # 2. Select columns for the matrix
    # We want to see how Daily_Fraud_Rate correlates with current and future prices
    correlation_cols = ['Daily_Fraud_Rate', 'Onion_Price'] + [f'Price_Lead_{lag}D' for lag in lag_days]
    corr_matrix = analysis_df[correlation_cols].corr()

    print("\n--- PEARSON CORRELATION MATRIX (Fraud vs. Future Prices) ---")
    print(corr_matrix[['Daily_Fraud_Rate']].sort_values(by='Daily_Fraud_Rate', ascending=False))
    
    # Identify the strongest lag
    strongest_lag_col = corr_matrix['Daily_Fraud_Rate'].drop('Daily_Fraud_Rate').idxmax()
    strongest_val = corr_matrix['Daily_Fraud_Rate'][strongest_lag_col]
    
    print(f"\nANALYSIS: The strongest predictive signal is '{strongest_lag_col}' (r = {strongest_val:.4f})")
    # --------------------------------------------------

    # 6. STATISTICAL EVALUATION
    print("\n" + "="*80)
    print("FORENSIC ANALYSIS: DOES ENFORCEMENT COLLAPSE PREDICT FUTURE PRICE SPIKES?")
    print("="*80)

    # Comparing Gold Standard vs Systemic Collapse on FUTURE prices
    gold_prices = analysis_df[analysis_df['Integrity_Tier'] == 'Gold Standard (0%)']['Price_Lead_3D'].dropna()
    collapse_prices = analysis_df[analysis_df['Integrity_Tier'] == 'Systemic Collapse (>15%)']['Price_Lead_3D'].dropna()

    print(f"Sample Size [Gold]: {len(gold_prices)} | Sample Size [Collapse]: {len(collapse_prices)}")

    if len(gold_prices) > 10 and len(collapse_prices) > 10:
        t_stat, p_val = stats.ttest_ind(gold_prices, collapse_prices, equal_var=False)
        
        avg_gold = gold_prices.mean()
        avg_collapse = collapse_prices.mean()
        diff = ((avg_collapse - avg_gold) / avg_gold) * 100

        print(f"\nAvg Price 3 Days After Honest Inspection:   Rs. {avg_gold:.2f}")
        print(f"Avg Price 3 Days After Systemic Fraud:     Rs. {avg_collapse:.2f}")
        print(f"Observed Inflationary Delta:               {diff:+.2f}%")
        print(f"\nT-Statistic: {t_stat:.4f} | P-Value: {p_val:.6f}")

        print("\n--- BRUTAL VERDICT ---")
        if p_val < 0.05:
            print("Verdict: SIGNIFICANT CAUSAL SIGNAL.")
            print("Proof: Magistrate fraud is a leading indicator of market price spikes.")
            print("Action: Deploy anti-hoarding squads 72 hours after 'Speedrunner' patterns emerge.")
        else:
            print("Verdict: NO CAUSAL LINK FOUND.")
            print("Observation: Prices are rising independently of magistrate behavior.")
            print("Action: Focus on Supply Chain/Logistics; Magistrate fraud is just bad HR, not market manipulation.")
    else:
        print("\n!!! WARNING: Insufficient Sample Size for T-Test.")
        print("The data is too clean or too dirty to find a comparison group.")

    # 7. EXPORT
    # FIX: Creating Processed directory if missing and fixing the typo in filename
    os.makedirs(processed_folder, exist_ok=True)
    output_file = os.path.join(processed_folder, "Statistical_Pillar2_Lagged_Correlation.csv")
    analysis_df.to_csv(output_file, index=False)
    
    print("\n" + "="*80)
    print(f"SUCCESS: Pillar 2 Complete. Lagged analysis exported to:\n{output_file}")
    print("="*80)

if __name__ == "__main__":
    execute_pillar_2_hardened()

>>> INITIALIZING PILLAR 2: LAGGED CORRELATION & ROBUST HYPOTHESIS TESTING...

>>> CALCULATING CROSS-LAG CORRELATION MATRIX...

--- PEARSON CORRELATION MATRIX (Fraud vs. Future Prices) ---
                  Daily_Fraud_Rate
Daily_Fraud_Rate          1.000000
Price_Lead_3D            -0.083286
Price_Lead_5D            -0.162666
Price_Lead_1D            -0.257296
Onion_Price              -0.279494
Price_Lead_7D            -0.429745

ANALYSIS: The strongest predictive signal is 'Price_Lead_3D' (r = -0.0833)

FORENSIC ANALYSIS: DOES ENFORCEMENT COLLAPSE PREDICT FUTURE PRICE SPIKES?
Sample Size [Gold]: 17 | Sample Size [Collapse]: 0

!!! WARNING: Insufficient Sample Size for T-Test.
The data is too clean or too dirty to find a comparison group.

SUCCESS: Pillar 2 Complete. Lagged analysis exported.


# Pillar 3: Correlation vs Causation (Separating Genuine Supply Shortages with Market Manipulation)

### 1. Is the price spike a "freak accident" or a real anomaly?

Prices naturally wiggle a little bit every day. The code now asks: **"What has the average price been for the last 7 days?"** * Instead of comparing today's price to yesterday's, it compares today’s price to the **weekly trend** (the Moving Average).

* It only raises an alarm if today's price is **more than 10% higher** than that weekly average. This filters out "noise" and focuses on major price hikes.

### 2. Is the market actually open and trading?

This code asks: **"Is the arrival of goods greater than zero?"**

* It ignores days with no trading, ensuring it only analyzes active market manipulation.

### 3. Is the supply increasing while prices "break away" from the norm?

The core "Hoarding" logic is now more strict. It asks: **"Even though more vegetables arrived today than yesterday, did the price somehow skyrocket far above the normal weekly trend?"**

* This is a much stronger indicator of a "Cartel" or "Hoarding" because there is no logical economic reason for a price to stay that high when supply is healthy and the recent trend was lower.

---


### How the "Risk Score" looks now

The "City Daily Hoarding Risk" is now a **Verified Risk**. If you see a "3" in this column, it means that for Onions, Potatoes, and Tomatoes, the prices are all significantly higher than their weekly averages, despite the fact that plenty of stock is arriving in the city.


In [ ]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

def execute_pillar_3_hoarding_detection():
    print(">>> INITIALIZING PILLAR 3: REFINED HOARDING DETECTION (SMOOTHING ENABLED)...")

    processed_folder = os.path.join('..', 'Data', 'Processed')
    
    # 1. Ingest the Data

    file_path = os.path.join(processed_folder, 'Master_Market_Fact.csv')
    
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"!!! CRITICAL ERROR: Could not load file at {file_path}. {e}")
        return

    # 2. Domain Knowledge Cleanup
    columns_to_drop = ['Wheat_Price', 'Wheat_Arrival', 'Banana_Price', 'Banana_Arrival']
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

    # 3. Standardize and Sort (Crucial for Rolling Windows)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['City', 'Date']).reset_index(drop=True)

    commodities = ['Onion', 'Potato', 'Tomato']
    
    # 4. The Refined Hoarding Algorithm (Rolling Window Logic)
    print(">>> Applying 7-Day Smoothing & Deviation Analysis...")
    
    for comm in commodities:
        price_col = f'{comm}_Price'
        arrival_col = f'{comm}_Arrival'
        
        # A. Calculate 7-Day Rolling Average Price per City
        # min_periods=3 ensures we still get a value even if a few days are missing
        df[f'{comm}_Price_7d_MA'] = df.groupby('City')[price_col].transform(
            lambda x: x.rolling(window=7, min_periods=3).mean()
        )

        # B. Calculate % Deviation from the Trend (Current Price vs. 7-Day Average)
        df[f'{comm}_Price_Dev_From_Trend'] = ((df[price_col] - df[f'{comm}_Price_7d_MA']) / df[f'{comm}_Price_7d_MA']) * 100
        
        # C. Calculate Day-over-Day Arrival Change
        df[f'{comm}_Arrival_Pct_Change'] = df.groupby('City')[arrival_col].pct_change() * 100
        
        # D. Refined Hoarding Condition:
        # 1. Supply Arrival stayed flat or INCREASED ( >= 0% change)
        # 2. Current Price is > 10% higher than the 7-day MOVING AVERAGE (Sustained Spike)
        # 3. Ensure Arrival > 0 (Ignores days where market was closed)
        hoarding_condition = (
            (df[f'{comm}_Arrival_Pct_Change'] >= 0) & 
            (df[f'{comm}_Price_Dev_From_Trend'] > 10.0) & 
            (df[arrival_col] > 0)
        )
        
        df[f'{comm}_Hoarding_Flag'] = np.where(hoarding_condition, 1, 0)

    # 5. Aggregate the Risk
    flag_cols = [col for col in df.columns if '_Hoarding_Flag' in col]
    df['City_Daily_Hoarding_Risk'] = df[flag_cols].sum(axis=1)
    
    # 6. Filter for Forensic Reporting
    # FIX: Changed variable from 'crimes_df' to 'anomalies_df' for professional tone
    anomalies_df = df[df['City_Daily_Hoarding_Risk'] > 0].copy()
    
    print("\n" + "="*80)
    print(f"SMOOTHED FORENSIC REPORT: ANOMALIES DETECTED")
    print("="*80)
    print(f"Total Trading Days Analyzed: {len(df)}")
    print(f"Verified Hoarding Anomalies (Deviated from Trend): {len(anomalies_df)}")
    
    if len(anomalies_df) > 0:
        print("\n>>> SAMPLE OF TREND-DEVIATING EVENTS (Price > 10% over 7-Day Avg):")
        view_cols = ['Date', 'City', 'Onion_Price', 'Onion_Price_7d_MA', 'Onion_Price_Dev_From_Trend', 'Onion_Hoarding_Flag']
        # Handle cases where Onion specific columns might not have anomalies in the top 5
        available_cols = [col for col in view_cols if col in anomalies_df.columns]
        print(anomalies_df[available_cols].head().round(2).to_string(index=False))
        
    # 7. Export the Fact Table
    # FIX: Routed to Processed folder with the correct, matching filename
    os.makedirs(processed_folder, exist_ok=True)
    output_file = os.path.join(processed_folder, "Statistical_Pillar3_Correlation_vs_Causation.csv")
    df.to_csv(output_file, index=False)
    
    print("\n" + "="*80)
    print(f"SUCCESS: Pillar 3 Refined. Output saved to:\n{output_file}")
    print("="*80)

if __name__ == "__main__":
    execute_pillar_3_hoarding_detection()

>>> INITIALIZING PILLAR 3: REFINED HOARDING DETECTION (SMOOTHING ENABLED)...
>>> Applying 7-Day Smoothing & Deviation Analysis...

SMOOTHED FORENSIC REPORT: ANOMALIES DETECTED
Total Trading Days Analyzed: 93
Verified Hoarding Anomalies (Deviated from Trend): 20

>>> SAMPLE OF TREND-DEVIATING EVENTS (Price > 10% over 7-Day Avg):
      Date       City  Onion_Price  Onion_Price_7d_MA  Onion_Price_Dev_From_Trend  Onion_Hoarding_Flag
2026-01-30 FAISALABAD         4000            3571.43                       12.00                    1
2026-01-31 FAISALABAD         4500            3785.71                       18.87                    1
2026-02-03 FAISALABAD         5000            4285.71                       16.67                    0
2026-02-04 FAISALABAD         5000            4500.00                       11.11                    1
2026-02-07 FAISALABAD         5000            4928.57                        1.45                    0

SUCCESS: Pillar 3 Refined. Output: Fact_Market_Hoar

# Pillar 4: Identifying Revenue Leakages through Z-Score

This code`s primary purpose is to identify which magistrates are performing poorly in collecting fines and to detect "KPI Gaming" where a magistrate might issue massive fines but fails to actually collect the money.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

def execute_pillar_4_dual_axis():
    print(">>> INITIALIZING PILLAR 4: DUAL-AXIS REVENUE AUDIT...")


    data_folder = os.path.join('..', 'Data')
    processed_folder = os.path.join(data_folder, 'Processed')
    
    # Point to the cleanly named database, NO MORE "New2"
    db_path = os.path.join(data_folder, 'PCCMD_Warehouse_New2.db')

    # FIX: Check if database exists before connecting to prevent blind crashes
    if not os.path.exists(db_path):
        print(f"!!! CRITICAL ERROR: Database not found at {db_path}")
        return

    conn = sqlite3.connect(db_path)

    query = """
        SELECT 
            Magistrate_ID, City,
            COUNT(Inspection_ID) as Total_Inspections,
            SUM(CASE WHEN Violation_Found = 'Yes' THEN 1 ELSE 0 END) as Total_Violations,
            SUM(Fine_Amount) as Total_Fine_Issued_Rs,
            SUM(CASE WHEN Fine_Status = 'Paid' THEN Fine_Amount ELSE 0 END) as Total_Fine_Collected_Rs
        FROM Fact_Inspections_Enriched
        GROUP BY Magistrate_ID, City;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    df.fillna(0, inplace=True)

    # Base Metrics
    df['Recovery_Rate'] = np.where(df['Total_Fine_Issued_Rs'] > 0, df['Total_Fine_Collected_Rs'] / df['Total_Fine_Issued_Rs'], 0)
    df['Avg_Fine_Amount'] = np.where(df['Total_Violations'] > 0, df['Total_Fine_Issued_Rs'] / df['Total_Violations'], 0)
    df['Unreconciled_Revenue_Rs'] = df['Total_Fine_Issued_Rs'] - df['Total_Fine_Collected_Rs']

    # --- SURGICAL CHANGE 1: Kill Survival Bias ---
    df['Low_Activity_Flag'] = np.where(df['Total_Violations'] < 10, 1, 0)

    # --- SURGICAL CHANGE 5: Administrative Friction Buffer (85% Target) ---
    target_rate = 0.85
    df['Expected_Collection_Rs'] = df['Total_Fine_Issued_Rs'] * target_rate
    df['Actionable_Leakage_Rs'] = np.maximum(0, df['Expected_Collection_Rs'] - df['Total_Fine_Collected_Rs'])

    # --- SURGICAL CHANGE 2: Absolute SLA Floor ---
    df['Absolute_Failure_Flag'] = np.where((df['Recovery_Rate'] < 0.75) & (df['Low_Activity_Flag'] == 0), 1, 0)

    # Compute City-Level Baselines ONLY on active magistrates
    active_df = df[df['Low_Activity_Flag'] == 0]
    city_stats = active_df.groupby('City').agg(
        City_Mean_Recovery=('Recovery_Rate', 'mean'),
        City_Std_Recovery=('Recovery_Rate', 'std'),
        City_Median_Fine=('Avg_Fine_Amount', 'median')
    ).reset_index()
    
    city_stats['City_Std_Recovery'] = city_stats['City_Std_Recovery'].replace(0, 0.01)
    df = pd.merge(df, city_stats, on='City', how='left')

    # --- SURGICAL CHANGE 3: Detect KPI Gaming ---
    # Flag if average fine is > 3x the city median AND recovery is catastrophic (< 10%)
    df['KPI_Gaming_Flag'] = np.where(
        (df['Avg_Fine_Amount'] > (3 * df['City_Median_Fine'])) & 
        (df['Recovery_Rate'] < 0.10) &
        (df['Low_Activity_Flag'] == 0), 
        1, 0
    )

    # Calculate Relative Z-Scores
    df['Recovery_Z_Score'] = np.where(
        df['Low_Activity_Flag'] == 0,
        (df['Recovery_Rate'] - df['City_Mean_Recovery']) / df['City_Std_Recovery'],
        0 
    )
    df['Severe_Variance_Flag'] = np.where(df['Recovery_Z_Score'] < -2.0, 1, 0)

    # --- SURGICAL CHANGE 4: Revenue-Weighted Priority Score ---
    df['Risk_Magnitude_Score'] = df['Actionable_Leakage_Rs'] * df['Recovery_Z_Score'].abs()

    # Clean up for Power BI
    df = df.round({'Recovery_Rate': 2, 'Avg_Fine_Amount': 0, 'City_Mean_Recovery': 2, 'City_Std_Recovery': 4, 'City_Median_Fine': 0, 'Recovery_Z_Score': 2, 'Risk_Magnitude_Score': 0, 'Actionable_Leakage_Rs': 0})
    
    # FIX: Dynamically route the output to the Processed folder
    os.makedirs(processed_folder, exist_ok=True)
    output_file = os.path.join(processed_folder, "Statistical_Pillar4_Revenue_Audit_Zscore.csv")
    df.to_csv(output_file, index=False)

    # Executive Output
    print("\n" + "="*80)
    print("EXECUTIVE AUDIT: THE DUAL-AXIS MODEL")
    print("="*80)
    print(f"Total Magistrates Audited: {len(df)}")
    print(f"Ghost Magistrates (Low Activity): {df['Low_Activity_Flag'].sum()}")
    print(f"SLA Absolute Failures (< 75%): {df['Absolute_Failure_Flag'].sum()}")
    print(f"KPI Gamers (Uncollectable Mega-Fines): {df['KPI_Gaming_Flag'].sum()}")
    
    high_risk = df[(df['Severe_Variance_Flag'] == 1) | (df['KPI_Gaming_Flag'] == 1)]
    if not high_risk.empty:
        print("\n>>> CRITICAL OFFENDERS (Sorted by Financial Impact):")
        high_risk = high_risk.sort_values(by='Risk_Magnitude_Score', ascending=False)
        view_cols = ['Magistrate_ID', 'City', 'Recovery_Rate', 'Avg_Fine_Amount', 'KPI_Gaming_Flag', 'Risk_Magnitude_Score']
        print(high_risk[view_cols].head().to_string(index=False))

    print("\n" + "="*80)
    print(f"SUCCESS: Exported {output_file}")
    print("="*80)

if __name__ == "__main__":
    execute_pillar_4_dual_axis()

>>> INITIALIZING PILLAR 4: DUAL-AXIS REVENUE AUDIT...

EXECUTIVE AUDIT: THE DUAL-AXIS MODEL
Total Magistrates Audited: 145
Ghost Magistrates (Low Activity): 0
SLA Absolute Failures (< 75%): 40
KPI Gamers (Uncollectable Mega-Fines): 0

>>> CRITICAL OFFENDERS (Sorted by Financial Impact):
Magistrate_ID       City  Recovery_Rate  Avg_Fine_Amount  KPI_Gaming_Flag  Risk_Magnitude_Score
      MAG-120     LAHORE           0.00          11021.0                0             2606121.0
      MAG-220 RAWALPINDI           0.00           8709.0                0             1920038.0
      MAG-303 FAISALABAD           0.63           8865.0                0              229119.0
      MAG-413     MULTAN           0.61           9775.0                0              213032.0

SUCCESS: Exported Dim_Magistrate_DualAxis_Profile.csv
